# CVX Colab runner — batch M2+M3 (E0–E4)

Thin runner: it only clones the repo, installs the lock file and calls `python -m cvx.run`.
All ML code lives in `src/cvx/`.

1. **Runtime → Change runtime type → CPU.**
2. Run the cells **top to bottom**. Cells 4–8 run one experiment each (speed mode: the whole batch must finish in one session in under 1 hour).
3. Each run writes a folder `FakeJobProject/cvx_runs/<experiment>/<UTC timestamp>/` on Drive. A run is finished only when its `env.json` says `"status": "complete"` (cell 9 lists them).
4. Copy the complete run folders to `runs/` on the laptop, then run `scripts/make_tables.py` and `scripts/make_figures.py` there.

In [ ]:
# Cell 1 — Mount Drive, get the code (private repo, token from Colab Secrets), check Python.
import base64
import json
import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive, userdata

assert sys.version_info[:2] == (3, 13), f"Expected Python 3.13, got {sys.version}"
drive.mount("/content/drive")

REPO_URL = "https://github.com/amareswarraok007/fakejobdetection_research.git"
REPO_DIR = Path("/content/fakejobdetection_research")
DATA = "/content/drive/MyDrive/FakeJobProject/data/raw/fake_job_postings.csv"
RUNS = "/content/drive/MyDrive/FakeJobProject/cvx_runs"


def run_git(args, cwd=None):
    """Run git with the token sent as an HTTP header. The token is never printed or stored."""
    token = userdata.get("GITHUB_TOKEN")
    header = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    command = ["git", "-c", f"http.extraHeader=Authorization: Basic {header}", *args]
    result = subprocess.run(command, cwd=cwd, capture_output=True, text=True)
    output = (result.stdout + result.stderr).replace(token, "***").replace(header, "***")
    if result.returncode != 0:
        raise RuntimeError(f"git {args[0]} failed:\n{output}")
    return output


if (REPO_DIR / ".git").exists():
    print(run_git(["pull", "--ff-only"], cwd=REPO_DIR))
else:
    run_git(["clone", REPO_URL, str(REPO_DIR)])
os.chdir(REPO_DIR)
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True)
print("Code at commit:", commit.stdout.strip())
Path(RUNS).mkdir(parents=True, exist_ok=True)

In [ ]:
# Cell 2 — Install the exact locked versions, the cvx package and the NLTK data (P-CV).
!pip install -q -r requirements-lock.txt
!pip install -q -e .
!python -m nltk.downloader -q stopwords wordnet
!python -c "import json, cvx.runio as r; print(json.dumps(r.library_versions(), indent=1))"

In [ ]:
# Cell 3 — Check the CSV on Drive is the file profiled in reports/data_profile.json.
expected = json.loads(Path("reports/data_profile.json").read_text())["dataset"]["sha256"]
check = subprocess.run(
    [sys.executable, "-c", f"from cvx.data.load import file_sha256; print(file_sha256({DATA!r}))"],
    capture_output=True, text=True, check=True,
)
print("Dataset matches the profile:", check.stdout.strip() == expected)
assert check.stdout.strip() == expected, "Different CSV: stop and tell the owner/agent."

In [ ]:
%%time
# Cell 4 — E0: reproduce Career Verify (P-CV, random 80/10/10, LR)
!python -m cvx.run --config configs/e0_reproduce.yaml --data "{DATA}" --runs-dir "{RUNS}"

In [ ]:
%%time
# Cell 5 — E1: reported (E1a/E1b, balanced before split) vs realistic (E1c)
!python -m cvx.run --config configs/e1_leakage.yaml --data "{DATA}" --runs-dir "{RUNS}"

In [ ]:
%%time
# Cell 6 — E2: corrected split with P-CV (E2a) and P-CVX (E2b)
!python -m cvx.run --config configs/e2_protocol.yaml --data "{DATA}" --runs-dir "{RUNS}"

In [ ]:
%%time
# Cell 7 — E3: NB, linear SVM, RF, LR, XGBoost (P-CVX, text only)
!python -m cvx.run --config configs/e3_models.yaml --data "{DATA}" --runs-dir "{RUNS}"

In [ ]:
%%time
# Cell 8 — E4: text vs text+metadata (LR, XGBoost); class weights vs SMOTE (LR)
!python -m cvx.run --config configs/e4_ablations.yaml --data "{DATA}" --runs-dir "{RUNS}"

In [ ]:
# Cell 9 — Which runs are complete? Copy the complete folders to runs/ on the laptop.
for env_file in sorted(Path(RUNS).glob("*/*/env.json")):
    env = json.loads(env_file.read_text())
    print(f"{env['status']:9}  {env_file.parent.relative_to(RUNS)}")